In [40]:
import pickle 

with open('dataset.pkl', 'rb') as f:
    data = pickle.load(f)

# Tokenization

We will use NLTK for this task , we will tokenize the all the Events into unique tokens

Now let's see how do we access these data.

In [41]:
data[0]['events'][1]

{'pitch': 87,
 'velocity': 100,
 'start': 0.38636349999999997,
 'end': 0.5170451875,
 'instrument': 40}

In [42]:
unique_composers = sorted(set(entry['composer'] for entry in data))
print(unique_composers)


['Bach', 'Beethoven', 'Brahms', 'Cambini', 'Dvorak', 'Faure', 'Haydn', 'Mozart', 'Ravel', 'Schubert']


In [ ]:
import numpy as np
import json

def tokenize_midi_with_note_on_off(data):
    tokens = []
    event_signatures = []

    for entry in data:
        composer = f"COMPOSER_{entry['composer']}"
        tokens.append(composer)

        events = []
        
        for event in entry['events']:
            pitch = f"PITCH_{event['pitch']}"
            velocity = f"VEL_{event['velocity']}"
            instrument = f"INST_{event['instrument']}"
            
            # Note On Event
            events.append({
                "type": "NOTE_ON",
                "start": event["start"],
                "pitch": pitch,
                "velocity": velocity,
                "instrument": instrument
            })

            # Note Off Event
            events.append({
                "type": "NOTE_OFF",
                "end": event["end"],
                "pitch": pitch,
                "velocity": velocity,
                "instrument": instrument
            })

        # Sort by time (NOTE_ON by start, NOTE_OFF by end)
        events.sort(key=lambda x: x.get("start", x.get("end", 0)))

        for e in events:
            if e["type"] == "NOTE_ON":
                token = (f"NOTE_ON_{e['pitch']}", e["velocity"], e["instrument"], f"START_{e['start']}")
            else:
                token = (f"NOTE_OFF_{e['pitch']}", e["velocity"], e["instrument"], f"END_{e['end']}")
            
            tokens.append(token)
            event_signatures.append(tuple(sorted(token)))

    # Create unique mapping
    unique_events = list(set(event_signatures))
    event_to_token = {str(event): idx for idx, event in enumerate(unique_events)}  # Convert tuple keys to strings
    
    # Generate tokenized data
    tokenized_events = [event_to_token[str(event)] for event in event_signatures]
    
    # Save as JSON
    with open("tokens.json", "w") as f:
        json.dump(tokens, f, indent=4)
    
    with open("tokenized_events.json", "w") as f:
        json.dump(tokenized_events, f, indent=4)
    
    with open("event_to_token.json", "w") as f:
        json.dump(event_to_token, f, indent=4)
    
    return tokens, tokenized_events, event_to_token


Using device: cpu
Processing input MIDI...
❌ Error loading MIDI file: [Errno 2] No such file or directory: 'input.mid'


In [44]:
tokenize_midi_with_note_on_off(data)

(['COMPOSER_Bach',
  ('NOTE_ON_PITCH_88', 'VEL_100', 'INST_40', 'START_0.25'),
  ('NOTE_OFF_PITCH_88', 'VEL_100', 'INST_40', 'END_0.3806816875'),
  ('NOTE_ON_PITCH_87', 'VEL_100', 'INST_40', 'START_0.38636349999999997'),
  ('NOTE_OFF_PITCH_87', 'VEL_100', 'INST_40', 'END_0.5170451875'),
  ('NOTE_ON_PITCH_88', 'VEL_100', 'INST_40', 'START_0.5227269999999999'),
  ('NOTE_OFF_PITCH_88', 'VEL_100', 'INST_40', 'END_0.7675186666666666'),
  ('NOTE_ON_PITCH_83', 'VEL_100', 'INST_40', 'START_0.7727269999999999'),
  ('NOTE_OFF_PITCH_83', 'VEL_100', 'INST_40', 'END_1.0175186666666667'),
  ('NOTE_ON_PITCH_80', 'VEL_100', 'INST_40', 'START_1.022727'),
  ('NOTE_OFF_PITCH_80', 'VEL_100', 'INST_40', 'END_1.2675186666666667'),
  ('NOTE_ON_PITCH_83', 'VEL_100', 'INST_40', 'START_1.272727'),
  ('NOTE_OFF_PITCH_83', 'VEL_100', 'INST_40', 'END_1.5175186666666667'),
  ('NOTE_ON_PITCH_76', 'VEL_100', 'INST_40', 'START_1.522727'),
  ('NOTE_OFF_PITCH_76', 'VEL_100', 'INST_40', 'END_1.6425186666666665'),
  ('NOT

In [ ]:
seq_length =  384,          # Increased but still memory-efficient
batch_size =  8,           # Increased from 2 for better training
d_model =  512,           # Restored to standard size
n_layers =  8,            # Increased layers
n_heads =  8,            # Increased attention heads
d_ff =  2048,           # Increased feed-forward size
dropout =  0.1,
learning_rate =  1e-4,
warmup_steps =  4000,
weight_decay =  0.01,    # L2 regularization
gradient_clip_val =  1.0